> 在线讲义：[第 3 章 · 日志、launch 和 bag：一次启动、看清日志、录下来重放](https://wikieden.github.io/ros2-from-scratch/lessons/03-launch-log-bag.html) · 本书《从零学 ROS 2》

# 第 3 章 · 日志、launch 和 bag：一次启动、看清日志、录下来重放

本章回答：节点一多，每个开一个终端太累，怎么一条命令全启动？几十个节点同时打印，怎么看日志？机器人出了怪事，怎么把当时的数据录下来，回到桌前重放复现？

配套讲义：仓库里 `ros2/notes/lessons/03-launch-log-bag.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 ROS 2。日志过滤、名字展开、录放这三件事的规则本身很简单，用几行 Python 就能看清；真命令（ros2 launch / ros2 bag）留到 Linux 上做里程碑 R-M1。

In [ ]:
import json

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 日志级别 | 只处理不低于设定级别的 | `passes(level, 'INFO')` | DEBUG 丢掉，其余打印 |
| 第 2 步 命名空间 | 相对名加前缀 | `resolve_name('turtle1/cmd_vel', 'turtlesim1')` | `/turtlesim1/turtle1/cmd_vel` |
| 第 2 步 绝对名 | `/` 开头不受影响 | `resolve_name('/rosout', ...)` | `/rosout` |
| 第 3 步 录 | 存序列化后的字节 | `record(raw)` | `(10.0, '/turtle1/cmd_vel', b'...')` |
| 第 3 步 放 | 按原节奏；可调速 | `play(bag)` / `play(bag, rate=2.0)` | 0, 0.5, 2.0 秒 / 0, 0.25, 1.0 秒 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `LEVELS`**

In [ ]:
LEVELS = ["DEBUG", "INFO", "WARN", "ERROR", "FATAL"]  # 从低到高

**函数 `passes`**：日志器只处理不低于设定级别的消息。

In [ ]:
def passes(level: str, threshold: str) -> bool:
    """日志器只处理不低于设定级别的消息。"""
    return LEVELS.index(level) >= LEVELS.index(threshold)

**函数 `play`**：放：按原来的时间间隔重新发出去；rate=2 就是两倍速，间隔减半。返回 (回放后第几秒发出, 话题, 消息)。

In [ ]:
def play(bag: list[tuple[float, str, bytes]], rate: float = 1.0) -> list[tuple[float, str, dict]]:
    """放：按原来的时间间隔重新发出去；rate=2 就是两倍速，间隔减半。返回 (回放后第几秒发出, 话题, 消息)。"""
    t0 = bag[0][0]
    return [(round((t - t0) / rate, 3), topic, json.loads(data)) for t, topic, data in bag]

**函数 `record`**：录：(时间, 话题, 消息) → (时间, 话题, 序列化后的字节)。录的时候不用反序列化，所以省 CPU。

In [ ]:
def record(messages: list[tuple[float, str, dict]]) -> list[tuple[float, str, bytes]]:
    """录：(时间, 话题, 消息) → (时间, 话题, 序列化后的字节)。录的时候不用反序列化，所以省 CPU。"""
    return [(t, topic, json.dumps(msg).encode()) for t, topic, msg in messages]

**函数 `resolve_name`**：名字展开：/ 开头是绝对名，原样；~ 开头是私有名，接在节点名后；其余是相对名，接在命名空间后。

In [ ]:
def resolve_name(name: str, namespace: str = "/", node: str = "") -> str:
    """名字展开：/ 开头是绝对名，原样；~ 开头是私有名，接在节点名后；其余是相对名，接在命名空间后。"""
    ns = "/" + namespace.strip("/") if namespace.strip("/") else ""
    if name.startswith("/"):
        return name
    if name.startswith("~"):
        return f"{ns}/{node}{name[1:]}"
    return f"{ns}/{name}"

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 日志级别：设成 INFO，DEBUG 就不打了")
for level in LEVELS:
    print(f"  {level:<5} 在 INFO 下 → {'打印' if passes(level, 'INFO') else '丢掉'}")

In [ ]:
print("\n第二步 · launch 的 namespace：相对名自动加前缀")
for ns in ["turtlesim1", "turtlesim2"]:
    print(f"  namespace={ns}: turtle1/cmd_vel → {resolve_name('turtle1/cmd_vel', ns)}")
print("  绝对名 /rosout 不受命名空间影响 →", resolve_name("/rosout", "turtlesim1"))
print("  私有名 ~/set_parameters（节点 sim）→", resolve_name("~/set_parameters", "turtlesim1", "sim"))

In [ ]:
print("\n第三步 · bag：录成字节，按原节奏放；rate=2 两倍速")
raw = [(10.0, "/turtle1/cmd_vel", {"linear": {"x": 2.0}}),
       (10.5, "/turtle1/cmd_vel", {"angular": {"z": 1.8}}),
       (12.0, "/turtle1/cmd_vel", {"linear": {"x": -2.0}})]  # 三条指令和时间是编的
bag = record(raw)
print("  录下的第一条:", bag[0])
print("  原速回放:", [(t, m) for t, _, m in play(bag)])
print("  两倍速回放的时间:", [t for t, _, _ in play(bag, rate=2.0)])

## 3. 代码里才看得到的

1. **私有名 `~`**：讲义第 2 步没展开。`~/xxx` 接在"命名空间 / 节点名"后面，第 2 章看到的 `/turtlesim/set_parameters` 就是 `~/set_parameters` 展开的结果。
2. **回放只保相对时间**：bag 里存的是录制时的绝对时间戳，回放从"现在"开始按间隔重新发；要让节点用 bag 里的时间，得配合第 11 章的仿真时间。
3. **三条指令和时间是编的**，只为看清录放保持间隔。

## 4. 自己改一改

- 把门槛改成 `WARN`，看哪些级别被丢掉；
- 在 `raw` 里再加一条 `/turtle1/pose` 的消息，看录放后时间顺序是否还对。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **日志**：五级（调试、信息、警告、错误、致命），三去处（终端、文件、`/rosout`）；高频回调要节流。
- **launch**：一张清单起全家；命名空间分你我。
- **bag**：record 录、info 看、play 放；录的是字节，存的是 MCAP；回放时关掉原来的发布者。